# Gas-turbine secondary-air system — solving a large flow network

This notebook loads the **large showcase** network `gas_turbine_large.yaml`
(saved from the FNetLibUI tool, model `fns-flow-network`), solves the steady mean
flow with **FNS**, looks at how the air is distributed across the cooling sinks,
and runs a linear **perturbation** analysis on top of the converged mean flow.

The case models a turbine's **secondary-air** (cooling / sealing) distribution:

    HP bleed (total-P) --\
                          JunctionStaticP (mix-plenum) --> contraction --> ...
    LP bleed (mass-flow) /

    ... --> LosslessSplitter (main-manifold) --> 3 sub-manifold legs,
            each metering air to several fixed-back-pressure sinks through
            orifices (IsentropicAreaChange), dump nozzles (+ SuddenAreaChange)
            and labyrinth seals (series LossElements).

Three things make it more than a big tree:

* a `LossElement` **cross-bridge** links sub-manifold-0 to sub-manifold-1, so the
  graph carries a **cycle**;
* one sink sits at a back pressure *above* the local static pressure, so it
  **ingests** air — an emergent **backflow** the solver finds on its own;
* six constant-area **`Duct`** sections sit on the transport runs (the two inlet
  feeds, the main manifold pipe, and each sub-manifold feed). A duct is **inert in
  the mean flow** — its length never enters the steady residual and its two edges
  share one area — so it leaves the converged mean flow untouched while giving the
  **perturbation** network real wave-propagation phase.

63 elements / 63 edges, fully subsonic. All figures use the shared plotly theme,
`fns.plotting`.

In [1]:
import sys, os, warnings
from collections import Counter

sys.path.insert(0, os.path.abspath(".."))   # make the `fns` package importable

import numpy as np
import yaml
import plotly.graph_objects as go

from fns.io import load_case
from fns.plotting import use_fns_theme, COLORWAY

use_fns_theme()   # register + activate the modern FNS plotly template

CASE = "gas_turbine_large.yaml"

# The file is large; summarise it rather than printing it whole.
doc = yaml.safe_load(open(CASE))
ntypes = Counter(n["type"] for n in doc["model"]["nodes"])
print(f"model id : {doc['model']['id']}")
print(f"nodes    : {len(doc['model']['nodes'])}    edges: {len(doc['model']['edges'])}")
for t, c in ntypes.most_common():
    print(f"   {t:<22} {c}")

model id : fns-flow-network
nodes    : 63    edges: 63
   LossElement            22
   PressureOutlet         15
   IsentropicAreaChange   9
   Duct                   6
   JunctionStaticP        4
   SuddenAreaChange       4
   TotalPressureInlet     1
   MassFlowInlet          1
   LosslessSplitter       1


## 1. Load and solve

`load_case` parses `model.globalAttributes` (gas + reference scales),
`model.nodes` (element `type` + `attributes`) and `model.edges` (port handles +
area) into a `Network`. `Network.solve()` compiles the immutable problem and runs
the damped Newton solve from a quiescent start — no regime or flow-direction is
prescribed.

In [2]:
net = load_case(CASE)
sol = net.solve()
prob = net.compile()

print("converged :", sol.converged)
print("iterations:", sol.iterations)
print("||R_hat|| :", f"{sol.residual_norm:.2e}")
print("elements  :", prob.n_nodes, "  edges:", prob.n_edges)

converged : True
iterations: 15
||R_hat|| : 3.70e-15
elements  : 63   edges: 63


## 2. The converged mean flow

Every edge carries the recovered state `(mdot, p, h_t, rho, u, T, c, M, p_t)`.
The whole network stays comfortably **subsonic** (the v1 scope); the fastest edges
are the metering orifices and dump-nozzle throats. The `Duct` edges simply repeat
the state of the edge they sit on — same `mdot`, same area — confirming they are
inert in the mean flow.

In [3]:
nn = prob.node_names          # element (node) labels
edges = net._edges            # list of (tail_elem, head_elem, area)

def edge_label(e):
    t, h, _a = edges[e]
    return f"{nn[t]} -> {nn[h]}"

fields = ["mdot", "M", "p", "p_t", "T"]
hdr = f"{'e':>3}  {'edge (tail -> head)':<38} " + " ".join(f"{f:>9}" for f in fields)
print(hdr); print("-" * len(hdr))
for e in range(prob.n_edges):
    s = sol.edge(e)
    row = " ".join(f"{s[f]:>9.4g}" for f in fields)
    print(f"{e:>3}  {edge_label(e):<38} " + row)

mmax = max(range(prob.n_edges), key=lambda e: abs(sol.edge(e)["M"]))
print(f"\nmax |M| = {abs(sol.edge(mmax)['M']):.3f}  on edge {mmax}  ({edge_label(mmax)})")

  e  edge (tail -> head)                         mdot         M         p       p_t         T
---------------------------------------------------------------------------------------------
  0  hp-bleed -> hp-feed-duct                   14.68    0.3295 8.348e+05     9e+05     685.1
  1  lp-bleed -> lp-feed-duct                       3   0.07511 8.348e+05 8.381e+05     479.5
  2  mix-plenum -> main-feed                    17.68    0.2585 8.348e+05 8.745e+05     653.9
  3  main-feed -> main-feed-pipe                17.68    0.3163 8.159e+05 8.745e+05     649.7
  4  main-manifold -> sub-feed-0                5.673    0.2043 8.494e+05 8.745e+05     657.2
  5  sub-feed-0 -> sub-feed-duct-0              5.673    0.2093 8.287e+05 8.544e+05     656.9
  6  sub-manifold-0 -> orf-1480--560          -0.1371  -0.01566 8.287e+05 8.289e+05       700
  7  orf-1480--560 -> loss-1480--560          -0.1371  -0.03134 8.283e+05 8.289e+05     699.9
  8  loss-1480--560 -> sink-1480--560         -0.1371   -0.0

## 3. Air distribution across the sinks (and the backflow)

The interesting output of a secondary-air model is *how the feed splits*. We pull
the boundary edges straight from the perturbation **terminals** (the solver's own
view of where the network meets the outside), check the global **mass balance**,
and chart each cooling sink's draw. One bar goes **negative**: `sink-1480--560`
sits at 830 kPa, above the local manifold static pressure, so it ingests air. The
solver discovers that backflow by itself; the outlet's `backflowTotalTemperature`
closes the energy balance for the ingested stream.

In [4]:
from fns.perturbation import find_terminals

terms = find_terminals(prob, sol.x)
inlets  = [t for t in terms if t.at_tail]
outlets = [t for t in terms if not t.at_tail]

def term_name(t):
    tl, hd, _a = edges[t.edge]
    return nn[tl] if t.at_tail else nn[hd]

m_in  = sum(sol.edge(t.edge)["mdot"] for t in inlets)
m_out = sum(sol.edge(t.edge)["mdot"] for t in outlets)
print(f"feed-in  (HP + LP) = {m_in:+.4f} kg/s")
print(f"sink-out (net)     = {m_out:+.4f} kg/s")
print(f"imbalance          = {m_in - m_out:+.2e} kg/s  (machine zero)")

sink_names = [term_name(t) for t in outlets]
sink_mdot  = np.array([sol.edge(t.edge)["mdot"] for t in outlets])
order = np.argsort(sink_mdot)
bars  = sink_mdot[order]
colors = [COLORWAY[4] if m < 0 else COLORWAY[0] for m in bars]

fig = go.Figure(go.Bar(
    x=bars, y=[sink_names[i] for i in order], orientation="h",
    marker_color=colors, text=[f"{m:+.2f}" for m in bars], textposition="outside",
))
fig.add_vline(x=0.0, line=dict(color="#52606d", width=1))
fig.update_layout(
    title="Cooling-sink mass flow  (red = ingestion / backflow)",
    xaxis_title=r"$\text{mass flow } \dot{m}\;[\mathrm{kg/s}]$", height=560, width=780, showlegend=False,
)
fig.show()

feed-in  (HP + LP) = +17.6803 kg/s
sink-out (net)     = +17.6803 kg/s
imbalance          = -3.55e-15 kg/s  (machine zero)


## 4. The whole network as a Sankey

Because FNS is a *network* solver, the natural picture of the solution is the flow
graph itself. We lay the elements out with the UI canvas coordinates (the
`uiAttributes` the tool saved) and size each link by `|ṁ|`. The six **ducts** show
up as short pass-through nodes on the feed runs; the lone **red** link is the
backflow edge, drawn in its physical direction (into the network).

In [5]:
pos = {n["id"]: n["position"] for n in doc["uiAttributes"]["nodes"]}
id_by_index = {int(n["attributes"]["index"]): n["id"] for n in doc["model"]["nodes"]}

n_elem = prob.n_nodes   # graph "nodes" are the flow elements
xs = np.array([pos[id_by_index[i]]["x"] for i in range(n_elem)], float)
ys = np.array([pos[id_by_index[i]]["y"] for i in range(n_elem)], float)

def norm(v, lo=0.04, hi=0.96):
    v = v - v.min()
    return lo + (hi - lo) * (v / v.max() if v.max() > 0 else v)

nx_, ny_ = norm(xs), norm(ys)

src, tgt, val, lcol = [], [], [], []
for e in range(prob.n_edges):
    t, h, _a = edges[e]
    m = sol.edge(e)["mdot"]
    if m >= 0.0:
        src.append(t); tgt.append(h); val.append(m);  lcol.append("rgba(45,113,194,0.45)")
    else:
        src.append(h); tgt.append(t); val.append(-m); lcol.append("rgba(214,69,65,0.75)")  # backflow

fig = go.Figure(go.Sankey(
    arrangement="snap",
    node=dict(label=[nn[i] for i in range(n_elem)],
              x=list(nx_), y=list(ny_), pad=6, thickness=9, color=COLORWAY[1]),
    link=dict(source=src, target=tgt, value=val, color=lcol),
))
fig.update_layout(
    title="Secondary-air mass-flow network  (Sankey; red link = backflow)",
    height=820, width=1040, font_size=10,
)
fig.show()

## 5. Perturbation analysis on a branched network

The compiled network also supports a linear **perturbation** analysis on top of
the mean flow (theory §12): three characteristics — two acoustic ($f$, $g$) and
the **entropy / convected** wave ($h$) — propagating on the converged state. The
constant-area ducts added in §0 are what give those waves somewhere to *propagate*:
each duct contributes an acoustic delay $L/(c\pm u)$ (and a convective delay $L/u$
for entropy), so the transfer entries pick up real frequency-winding phase.

On a single duct chain you would read off a 2×2 (or 3×3) **transfer matrix**
between two edges. Here that does **not** apply between, say, the HP inlet and a
sink: they straddle the splitter and the sub-manifold junctions, so the wave state
at one edge does *not* determine the other. The library refuses to pretend
otherwise and points you at the rigorous descriptors — the **multiport scattering
matrix** over all terminals, and **source attribution** (what reaches one edge
from each terminal). We force the operator **once per frequency** and re-use that
solve for both.

In [6]:
from fns.perturbation import (
    perturbation_response, verify_perturbation, TransferMatrixWarning,
)

verify_perturbation(prob, sol.x)   # subsonic, flow-aligned, lengths > 0 -> OK

freq = np.linspace(20.0, 1500.0, 400)   # Hz
resp = perturbation_response(prob, sol.x, freq, excite=("acoustic", "entropy"))

# A naive 2-port between the HP inlet and a sink is non-physical here -- show it.
HP, SINK = 0, 54   # 'hp-bleed' inlet edge,  outlet edge feeding 'sink-1480-1320'
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", TransferMatrixWarning)
    resp.transfer_matrix(HP, SINK)
print("2-port inlet->sink :", caught[0].category.__name__ if caught else "no warning")

# The rigorous whole-network descriptor instead:
Smp = resp.multiport_scattering_matrix()
inc_lab, out_lab = resp.multiport_scattering_labels()
print(f"multiport scattering: {np.asarray(Smp).shape}  "
      f"({len(inc_lab)} incoming -> {len(out_lab)} outgoing waves)")

2-port inlet->sink : TransferMatrixWarning
multiport scattering: (400, 31, 20)  (20 incoming -> 31 outgoing waves)


### Source attribution at one sink

Rather than the full $31\times20$ multiport matrix, the readable cut is: of the
waves **leaving** `sink-1480-1320`, how much originates at each terminal, versus
frequency? Each outgoing characteristic ($f$, $g$, $h$ at that edge) is broken
down over the incoming terminal waves — all from the single forced solve above.
The phase winding now comes from the duct propagation upstream of the sink.

In [7]:
fig = resp.plot_contributions(
    SINK, freq, normalize="auto",
    title=f"What reaches {nn[edges[SINK][1]]} — contribution by source terminal",
)
fig.show()

The mean-flow split (§3–4) and the perturbation picture (§5) both come from the
**same** converged state and Jacobian. The steady solve finds the air
distribution — including the emergent backflow sink and the flow through the
cross-bridge cycle — with no regime switching, and the ducts leave it untouched.
The perturbation layer then rides that mean flow, using the ducts for propagation
phase, and on this branched topology it answers the questions that actually make
sense: the multiport scattering matrix over all terminals and the per-terminal
source attribution at any edge of interest.